In [ ]:
"""pip install -r requirements.txt"""

This file first runs all the tests and than the main function, the purpose is to get a step-by-step walk through of the centire code base, where the results are interactivly achived. It does the same thing as main.py, but used for debugging/education.

Run this from the terminal (while in the project folder):

.venv/bin/python -m pytest tests/ -v

In [ ]:
!.venv/bin/python -m pytest tests/ -v

In [ ]:
# conftest.py includes pytest fixtures, which won't auto-inject in a notebook
# --> hey are created manually
from pathlib import Path
BASE_DIR = Path()
STAFF_DIR = BASE_DIR / "staff_configs"
DEPT_FILE = BASE_DIR / "department_requirements.txt"
LAW_FILE = BASE_DIR / "swedish_work_law.txt"

In [ ]:
# Parse the inputs using config_parser (no LLM functionality, fallback text parser)
from src import config_parser
from datetime import date
import dataclasses
import pprint

start_date = date(2026, 4, 6)

do_you_want_to_use_llm = True
if do_you_want_to_use_llm == True:
    # Parse the inputs using config_parser (manual LLM functionality)
    parsed_inputs = config_parser.parse_all_inputs(
        staff_config_dir = STAFF_DIR,
        dept_req_file = DEPT_FILE,
        law_file = LAW_FILE,
        start_date = start_date,
        use_llm = False,                                # For now it has to be False as no Anthropic account has been created
        manual_llm = True, 
    )
else:
    # Parse the inputs using config_parser fallback text parser
    parsed_inputs = config_parser.parse_all_inputs(
        staff_config_dir = STAFF_DIR,
        dept_req_file = DEPT_FILE,
        law_file = LAW_FILE,
        start_date = start_date,
        use_llm = False,                                # For now it has to be False as no Anthropic account has been created
        manual_llm = False, 
    )

pprint.pprint(dataclasses.asdict(parsed_inputs))

In [ ]:
from src import solver

schedule = solver.build_schedule(
    inputs=parsed_inputs,
    start_date = start_date,
    num_weeks = 3,
    solver_time_limit_s = 60*5,
)

In [ ]:
print(f"assignments: {schedule.assignments}\n")
print(f"start_date: {schedule.start_date}\n")
print(f"num_weeks: {schedule.num_weeks}\n")
print(f"infeasible_preferences: {schedule.infeasible_preferences}\n")
print(f"solver_status: {schedule.solver_status}\n")
print(f"objective_value: {schedule.objective_value}\n")

In [ ]:
from src import exporters
exporters.export_excel(
    schedule = schedule,
    inputs = parsed_inputs,
    output_dir = BASE_DIR
)

In [ ]:
from src import reporter
reporter.generate_report(
    schedule = schedule,
    inputs = parsed_inputs,
    manual_llm = True
)